In [1]:
import pandas as pd

df = pd.read_csv('google_trends_raw.csv')
df = df.rename(columns={'Unnamed: 0': 'date'})
df['date'] = pd.to_datetime(df['date'])
print(f"Raw shape: {df.shape[0]} rows × {df.shape[1]} cols")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}\n")

# ── 1. Drop zero-signal columns ───────────────────────────────────────────────
# Columns where max value across all 47 days never exceeded 10
# These keywords had no meaningful search activity in this period
keyword_cols = [c for c in df.columns if c != 'date']
maxes = df[keyword_cols].max()
zero_signal = maxes[maxes <= 10].index.tolist()

print(f"[1] Dropping {len(zero_signal)} zero-signal columns (max ≤ 10 across all days):")
for col in zero_signal:
    print(f"    — {col} (max: {maxes[col]})")
df = df.drop(columns=zero_signal)
print(f"    Remaining: {len(df.columns) - 1} keyword columns\n")

# ── 2. Drop contextually irrelevant columns ───────────────────────────────────
# These keywords capture general background noise, not geopolitical event signal
# e.g. "sensex today" / "nifty today" are daily habit searches, not panic searches
# "india gdp", "reliance share price" are evergreen queries unrelated to the event
irrelevant = [
    'sensex today',          # daily routine search, not event-driven
    'nifty today',           # same
    'nifty prediction',      # generic, not event-driven
    'stock market india',    # too broad, always present
    'reliance share price',  # not in our stock universe, evergreen query
    'india gdp',             # macro background, not war-sensitive
    'import export india',   # too broad
    'best stocks to buy now',# retail noise, not event-driven
    'safe investment india', # too generic
    'sell stocks now',       # low signal (max was low)
    'indigo airlines',       # company-specific brand search, not sentiment
    'ongc share price',      # stock-specific price lookup, not sentiment
]
# Only drop if still present after step 1
irrelevant = [c for c in irrelevant if c in df.columns]
print(f"[2] Dropping {len(irrelevant)} contextually irrelevant columns:")
for col in irrelevant:
    print(f"    — {col}")
df = df.drop(columns=irrelevant)
print(f"    Remaining: {len(df.columns) - 1} keyword columns\n")

# ── 3. Validate date continuity ───────────────────────────────────────────────
date_range = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
missing_dates = date_range.difference(df['date'])
print(f"[3] Date continuity check:")
print(f"    Expected days: {len(date_range)}")
print(f"    Actual rows  : {len(df)}")
print(f"    Missing dates: {len(missing_dates)}")
if len(missing_dates):
    for d in missing_dates:
        print(f"    — {d.date()}")
print()

# ── 4. Check for nulls ────────────────────────────────────────────────────────
nulls = df.drop(columns='date').isnull().sum().sum()
print(f"[4] Null values: {nulls}")
print()

# ── 5. Export ─────────────────────────────────────────────────────────────────
keyword_cols_final = [c for c in df.columns if c != 'date']
df = df[['date'] + sorted(keyword_cols_final)]  # alphabetical for readability
df = df.sort_values('date').reset_index(drop=True)

df.to_csv('trends_clean.csv', index=False)
print(f"Saved: trends_clean.csv — {df.shape[0]} rows × {df.shape[1]} cols")
print(f"Keywords kept: {len(keyword_cols_final)}")
print(f"\nFinal columns:")
for col in keyword_cols_final:
    print(f"  {col}")

Raw shape: 47 rows × 61 cols
Date range: 2026-02-28 → 2026-04-15

[1] Dropping 20 zero-signal columns (max ≤ 10 across all days):
    — israel bombing (max: 8)
    — oil tanker attack (max: 2)
    — petrol price india (max: 10)
    — fuel price india (max: 5)
    — opec oil cut (max: 1)
    — nifty crash (max: 2)
    — sensex crash (max: 1)
    — nifty fall today (max: 2)
    — stock market crash india (max: 2)
    — should i sell stocks (max: 1)
    — market circuit breaker (max: 3)
    — rupee fall (max: 2)
    — rbi rate hike (max: 0)
    — spicejet shares (max: 1)
    — defence stocks india (max: 0)
    — hal share price (max: 0)
    — shipping stocks india (max: 0)
    — gold price india (max: 7)
    — where to invest during war (max: 0)
    — mutual fund safe (max: 7)
    Remaining: 40 keyword columns

[2] Dropping 12 contextually irrelevant columns:
    — sensex today
    — nifty today
    — nifty prediction
    — stock market india
    — reliance share price
    — india gdp
   